# Modelling — FX Activation

One notebook per product. A thin caller of `src/run.py` plus inspection. All logic — FLAML fit, external OOT evaluation, leakage report, card, score log — lives in `src/`; the notebook runs it and eyeballs the artifacts.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load
cfg.product, str(cfg.obs_date), len(cfg.features)

### 0 · (Optional) smoke-test the plumbing
Before the real run, confirm FLAML + metrics work on throwaway synthetic data. This makes no domain claim — it only checks the machinery.

In [ ]:
import numpy as np, pandas as pd
from flaml import AutoML
rng = np.random.default_rng(0); n = 4000
Xs = pd.DataFrame({f'f{i}': rng.normal(size=n) for i in range(6)})
ys = (Xs['f0'] + 0.3 * rng.normal(size=n) > 0).astype(int)
m = AutoML()
m.fit(X_train=Xs, y_train=ys, task='classification', metric='roc_auc',
      time_budget=10, estimator_list=['lgbm'], eval_method='cv', n_splits=3, verbose=0)
print('best estimator:', m.best_estimator, '| plumbing OK')

### 1 · The real run
Loads train/OOT/infer months (pruned in Spark), builds labels from the materialised target column, asserts the contract, writes the leakage report, fits FLAML on the curated features only, evaluates OOT against the population-base-rate baseline, scores the inference month, and writes the frozen config + card + score log to `artifacts/`.

In [ ]:
from src.run import run
result = run(cfg, spark)
result['run_id'], result['out']

### 2 · OOT metrics vs baseline
`oot_auc` is the headline; `top_decile_lift` over the base rate is what says whether the scores are useful for targeting (baseline is the population base rate, so AUC's reference is 0.5 and lift's is 1.0).

In [ ]:
result['metrics']

### 3 · Inspect what the run wrote

In [ ]:
import pandas as pd, json, pathlib
out = pathlib.Path(result['out'])
pd.read_csv(out / 'leakage_report.csv').head(10)

In [ ]:
json.loads((out / 'model_card.json').read_text())['caveats']

In [ ]:
scores = pd.read_parquet(out / 'scores.parquet')
print(scores.shape); scores.head()

### 4 · Before trusting anything, eyeball
1. Base rate plausible for the eligible population?  2. `top_decile_lift` meaningfully > 1?  3. Any `suspected_leak` feature that survived into the model — back to Feature Checks.  4. Single-month seasonality caveat in the card: don't over-trust the level.